# 16 - Job–Skill Bipartite Graph Construction

## Overview

This notebook converts the outputs of Chapter 1 into a graph-based representation of the job market. Starting from the skill probability matrix (jobs × skills), we construct a **job–skill bipartite graph** in which jobs and skills are represented as distinct node types, and edges encode whether a job requires a given skill. Edge weights store the predicted probability that the skill is required, preserving the full richness of the Chapter 1 outputs.

The purpose of this module is **structural**, not analytical. No embeddings, clustering, or downstream modelling are performed here. The sole objective is to build a correct, interpretable, and reusable graph object that will serve as the foundation for all subsequent Chapter 2 analyses.

## Data Inputs

The primary input is the saved **skill probability matrix**, where each row corresponds to a job posting and each column corresponds to one of the 27 skill groups. Each cell contains a value in \([0, 1]\), representing the model-estimated probability that the job requires that skill. The row index of this matrix is preserved and used as the unique job identifier in the graph.

## Graph Design

The graph is a **bipartite, undirected graph** with two node types:
- **Job nodes**: one node per job record, identified by the job index.
- **Skill nodes**: one node per skill group, identified by the skill name.

Edges connect jobs to skills under a single, explicit rule: an edge is added if the skill probability for that job exceeds a fixed threshold. Each edge carries a `weight` attribute equal to the original probability value, allowing the graph to retain graded information rather than binary skill flags.

This design ensures that jobs sharing many high-probability skill edges are structurally close in the graph, while skills that frequently co-occur across jobs become naturally connected through shared neighbours.

## Validation and Sanity Checks

After construction, the graph is validated through basic but non-negotiable checks:
- node counts match the expected number of jobs and skills,
- the total number of edges is plausible (neither empty nor fully dense),
- spot checks confirm that edge weights match the original probability matrix and fall within valid bounds.

These checks ensure that the graph faithfully represents the Chapter 1 outputs without distortion or misalignment.

## Outputs and Persistence

The final graph is saved as a model artefact for reuse in later modules. Because the graph encodes a non-trivial structural transformation of the data, it is treated as a durable intermediate object rather than something to be rebuilt repeatedly. A small metadata record is saved alongside the graph to document the edge rule, threshold choice, and data provenance.


* Edge rule: thresholded weighted edges
* Threshold: 0.5
* Edge weight: skill probability

## Set up

### Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path
import networkx as nx
import pickle

### Path

In [57]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

### Compute Skill Prob Matrix

In [58]:
from src.job_intel.models.skill_prob_matrix import build_skill_probability_matrix
from src.job_intel.config import CH1_PROCESSED_SALARY_MODEL_PCA_DF

In [59]:
df = pd.read_csv(CH1_PROCESSED_SALARY_MODEL_PCA_DF)

prob_mat = build_skill_probability_matrix(jobs_df= df)

## Save and test the indices

In [60]:
df['job_id'] = range(len(df))
df = df.set_index("job_id")
from src.job_intel.config import PROCESSED_DATA_DIR
df.to_csv(PROCESSED_DATA_DIR / 'salary_model_dfv03_pca_jobid.csv', index=True)

From this point onward, we introduce and persist an explicit job_id index to serve as a stable identifier across all Chapter 2 artefacts and downstream analyses. This step is taken as a forward-looking safety measure rather than to correct a detected error. Extensive diagnostics confirmed that all prior transformations preserved row order deterministically, and that any discrepancies between datasets were caused exclusively by intentional row filtering rather than index misalignment or reordering. As a result, earlier use of an autogenerated index had no practical consequences for model training or evaluation. Modifying the upstream Chapter 0 or Chapter 1 pipelines retroactively would therefore provide no additional correctness guarantees, while introducing unnecessary complexity and risk. Freezing a stable job index at this stage ensures unambiguous alignment for graph construction, embeddings, and clustering, without disturbing validated components of the existing pipeline.

In [61]:
from src.job_intel.config import CH1_PROCESSED_SALARY_MODEL_DF, CH1_PROCESSED_SALARY_MODEL_PCA_DF, CH0_PROCESSED_JOBS_FILE

t1 = df
t2 = pd.read_csv(CH1_PROCESSED_SALARY_MODEL_DF)
t3 = pd.read_csv(CH1_PROCESSED_SALARY_MODEL_PCA_DF)
t4 = pd.read_csv(CH0_PROCESSED_JOBS_FILE)
t4 = t4.dropna(subset=['sal_mean'])

def order_match_ratio(df_ref, df_other, col):
    n = min(len(df_ref), len(df_other))
    return (df_ref[col].to_numpy()[:n] == df_other[col].to_numpy()[:n]).mean()

for name, df_ in {
    "t2": t2,
    "t3": t3,
    "t4": t4,
}.items():
    r = order_match_ratio(t1, df_, "Job Description")
    print(f"Order match t1 vs {name}: {r:.4f}")


Order match t1 vs t2: 1.0000
Order match t1 vs t3: 1.0000
Order match t1 vs t4: 1.0000


In [62]:
# Make sure the indices are conserved
df.index.equals(prob_mat.index)

True

## Graph

In [63]:
# NODES

G = nx.Graph()

for i in prob_mat.index:
    G.add_node(i, bipartite = 'job')

for s in prob_mat.columns:
    G.add_node(s, bipartite = 'skill')

In [64]:
G.number_of_nodes() == len(df) + 27

True

In [38]:
# EDGES
for i in prob_mat.index:
    for s in prob_mat.columns:
        prob = prob_mat.loc[i, s]
        if prob >= 0.5: # Threshold inclusion
            G.add_edge(i, s, weight=prob)

In [39]:
G.number_of_edges()

40313

In [40]:
list(G.edges(data=True))[:5]

[(0,
  'core_programming__basic_prob',
  {'weight': np.float64(0.9967883881136653)}),
 (0,
  'data_engineering_pipelines__intermediate_prob',
  {'weight': np.float64(0.903640375749317)}),
 (0,
  'analytics_stats__basic_prob',
  {'weight': np.float64(0.9494952209418731)}),
 (0, 'bi_viz__intermediate_prob', {'weight': np.float64(0.9927902348471053)}),
 (0, 'db_storage__basic_prob', {'weight': np.float64(0.9955334554604361)})]

### Save the graph

In [41]:
from src.job_intel.config import MODELS_DIR
with open(MODELS_DIR / "job_skill_bipartite_thres0_5.gpickle", "wb") as f:
    pickle.dump(G, f)


In [42]:
# To loaded it later use

#with open(MODELS_DIR / "job_skill_bipartite_thres0_5.gpickle", "wb", "rb") as f:
#    G = pickle.load(f)

# === End of Notebook ===